initiate checkpoint, tokenizer, datacollator and dataset

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("glue", "mrpc")
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)


def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)


tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

postprocess dataset: remove redundant columns, reformat

In [2]:
tokenized_datasets = tokenized_datasets.remove_columns(["sentence1", "sentence2", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels") #labels provided → automatically computes and returns the loss.
tokenized_datasets.set_format("torch")
tokenized_datasets["train"].column_names

['labels', 'input_ids', 'token_type_ids', 'attention_mask']

In [3]:
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=8, collate_fn=data_collator
)
eval_dataloader = DataLoader(
    tokenized_datasets["validation"], batch_size=8, collate_fn=data_collator
)

#check
for batch in train_dataloader:
    break
print({k: v.shape for k, v in batch.items()})
len(train_dataloader)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'labels': torch.Size([8]), 'input_ids': torch.Size([8, 68]), 'token_type_ids': torch.Size([8, 68]), 'attention_mask': torch.Size([8, 68])}


459

In [4]:
#example output
outputs = model(**batch)
print(outputs.loss, outputs.logits.shape)
"""
tensor(0.9802, grad_fn=<NllLossBackward0>) torch.Size([8, 2])
we get 8 results with the logits for 2 labels each
"""

tensor(0.8913, grad_fn=<NllLossBackward0>) torch.Size([8, 2])


'\ntensor(0.9802, grad_fn=<NllLossBackward0>) torch.Size([8, 2])\nwe get 8 results with the logits for 2 labels each\n'

In [ ]:
from torch.optim import AdamW
#back propagation -> adjust weight with adam optimizer
optimizer = AdamW(model.parameters(), lr=5e-5)
"""
Backpropagation computes gradients
Adam uses those gradients to adjust weights
"""

In [6]:
from transformers import get_scheduler

num_epochs = 3
num_training_steps = num_epochs * len(train_dataloader) #total training step = #epochs * #batches
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

=============== The training loop ===============

extra note:
Modern Training Optimizations: To make your training loop even more efficient, consider:

Gradient Clipping: Add `torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)` before `optimizer.step()`

Mixed Precision: Use `torch.cuda.amp.autocast()` and `GradScaler` for faster training

Gradient Accumulation: Accumulate gradients over multiple batches to simulate larger batch sizes

Checkpointing: Save model checkpoints periodically to resume training if interrupted

In [7]:
#hardware check
import torch

device = torch.device("cuda")
model.to(device) #move model to GPU

#pretty progress bar
from tqdm.auto import tqdm

progress_bar = tqdm(range(num_training_steps))

model.train() #enable train mode
for epoch in range(num_epochs):
    for batch in train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()} #move each batch to GPU
        outputs = model(**batch) #foward pass
        loss = outputs.loss
        loss.backward() #back propagation

        optimizer.step() #adjust weight
        lr_scheduler.step() #linearly decrease the lr
        optimizer.zero_grad() #reset the gradient
        progress_bar.update(1) #459 batches * 3 epochs = 1377 steps

  0%|          | 0/1377 [00:00<?, ?it/s]

============= Evaluation loop ===============

In [ ]:
!pip install evaluate

In [ ]:
import evaluate
metric = evaluate.load("glue", "mrpc")
model.eval()
for batch in eval_dataloader:
  batch = {k: v.to(device) for k, v in batch.item()}
  output = model(**batch)
  preds = torch.argmax(output.logits, dim=-1)
  metric.add_batch(predictions=preds, references=batch["labels"])
metric.compute()